# PEFT multi-epoch manifest

In [ ]:
# 24GB card expected (bf16 7B all-linear). This is the real regime, not a tiny smoke.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('manage.py') and not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

In [ ]:
!pip install -q "transformers==5.12.1" "peft==0.18.0" "accelerate==1.14.0" \
  "sentence-transformers==5.5.1" "sacrebleu==2.6.0" "PyYAML==6.0.3"

## Step A — build the single anchor-cell config 

In [ ]:
import yaml
from src.peft.sweep import build_cell_config

base = yaml.safe_load(open("configs/peft_sweep.yaml"))
anchor = next(c for c in base["sweep"]["grid"] if c.get("anchor"))
cfg, out_dir = build_cell_config(base, anchor, base["sweep"]["output_base"])

print("anchor cell:", anchor, "-> output_dir:", out_dir)
t = cfg["peft"]["train"]
print(f"epochs={t['num_train_epochs']}  data.limit={cfg['data']['limit']}  "
      f"load_best={t.get('load_best_model_at_end')}  save_total_limit={t.get('save_total_limit')}")
assert t["num_train_epochs"] == 3,          "expected the real 3-epoch count"
assert cfg["data"]["limit"] is None,        "expected full data (no smoke limit)"
assert t.get("load_best_model_at_end") is False, "manifest is only written when load_best is false"

with open("configs/peft_anchor_e3.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("wrote configs/peft_anchor_e3.yaml")

## Step B 


In [ ]:
!python -m src.peft.train --config configs/peft_anchor_e3.yaml

## Step C — the manifest check 

In [ ]:
import json
from pathlib import Path

man_path = Path("models/peft_lora_r16_lr2e-4/epoch_checkpoints.json")
print(man_path.read_text())          # the `cat`

man = json.loads(man_path.read_text())
epochs = [m["epoch"] for m in man]
steps  = [m["step"] for m in man]
print("epochs:", epochs, " steps:", steps)

assert len(man) == 3,          f"expected 3 epoch entries, got {len(man)} -- multi-epoch manifest broken"
assert epochs == [1, 2, 3],    f"epochs not distinct/ordered 1..3: {epochs}"
assert len(set(steps)) == 3,   f"duplicate checkpoint step -- the e3-duplicate bug is back: {steps}"
assert all(Path(m["checkpoint"]).exists() for m in man), "a manifest checkpoint dir is missing on disk"
print("\nMANIFEST OK: 3 distinct epoch checkpoints, no duplicate e3 -- snapshot+dedup holds on a real run.")

## Step D — eval_loss trajectory 

In [ ]:
losses = [(m["epoch"], m["eval_loss"]) for m in man]
print("eval_loss by epoch:")
for ep, el in losses:
    print(f"  epoch {ep}: {el:.4f}")

vals = [el for _, el in losses]
assert len(set(round(v, 6) for v in vals)) > 1, \
    "eval_loss identical across epochs -- training is not updating the model"

print(f"\ndelta  e1->e2: {vals[1]-vals[0]:+.4f}   e2->e3: {vals[2]-vals[1]:+.4f}")
if vals[1] < vals[0]:
    tail = "e2->e3 rose = overfitting onset (expected, fine)." if vals[2] > vals[1] \
           else "e2->e3 still dropping."
    print("e1->e2 dropped (learning). " + tail)
else:
    print("WARNING: eval_loss did not drop e1->e2 -- inspect LR / masking / grads before the full sweep.")